# الإحصاء باستخدام R

اختبار الفرضيات، والانحدار، وفترات الثقة باستخدام مجموعات بيانات R المدمجة.

لا حاجة لتنزيل بيانات أو تثبيت حزم — يستخدم مكتبات R الأساسية فقط.

## 1. الإحصاء الوصفي

In [ ]:
data(mtcars)
cat("مجموعة البيانات: mtcars (", nrow(mtcars), " سيارات، ", ncol(mtcars), " متغيرات)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. اختبار t لعينتين

هل تحقق السيارات ذات ناقل الحركة اليدوي كفاءة استهلاك وقود (MPG) أفضل من السيارات الأوتوماتيكية؟

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("أوتوماتيكي:", round(mean(auto), 1), "ميل لكل غالون (n =", length(auto), ")\n")
cat("يدوي:   ", round(mean(manual), 1), "ميل لكل غالون (n =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nالاستنتاج:",
    ifelse(t_result$p.value < 0.05,
           "رفض H0 — تحقق السيارات اليدوية كفاءة MPG أعلى بشكل ملحوظ",
           "الفشل في رفض H0"))

## 3. اختبار كاي تربيع (Chi-Squared)

هل عدد الأسطوانات ونوع ناقل الحركة مستقلان؟

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("أوتوماتيكي", "يدوي")
print(tab)
cat("\n")
chisq.test(tab)

## 4. الانحدار الخطي المتعدد

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. تشخيصات الانحدار

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. فترات الثقة

In [ ]:
ci <- confint(model, level = 0.95)
cat("فترات الثقة 95%:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "التقدير", ylab = "",
     main = "فترات الثقة 95% للمعاملات")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. تحليل التباين أحادي الاتجاه (One-Way ANOVA)

هل يختلف MPG بشكل كبير باختلاف عدد الأسطوانات؟

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nمقارنات توكي (Tukey HSD) البعدية:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG حسب عدد الأسطوانات",
        xlab = "الأسطوانات", ylab = "ميل لكل غالون",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## الملخص

- **اختبار ويلش t**: في المقارنة أحادية الجانب غير المعدلة، تمتلك السيارات اليدوية متوسط MPG أعلى
- **كاي تربيع**: يشير جدول التوافق إلى وجود ارتباط، ولكن التكرارات المتوقعة الصغيرة تطلق تحذير تقريب، لذا يجب تفسير هذه النتيجة بحذر
- **الانحدار**: يعد الوزن وقوة الأحصنة مؤشرين سلبيين ذوي دلالة إحصائية بعد التعديل؛ ونوع ناقل الحركة ليس ذا دلالة في هذا النموذج
- **تحليل التباين ANOVA**: يختلف MPG بشكل ملحوظ بين مجموعات 4 و6 و8 أسطوانات؛ وتحدد نتائج توكي (Tukey) الفروق الثنائية